<a href="https://colab.research.google.com/github/Menezes-Gus/Estudos/blob/'dev'/AuxilioProvaCientista.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Data Science Toolkit - Abrangendo Estatística, Machine Learning e Pesquisa Operacional
Autor: GPT-5 (2025)
"""

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import linprog
from sklearn import metrics, model_selection, preprocessing, decomposition, feature_selection
from sklearn.linear_model import LogisticRegression, LinearRegression, LassoCV, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# =============================================================================
# ESTATÍSTICA BÁSICA
# =============================================================================

def describe_stats(series):
    """Retorna estatísticas básicas de uma série numérica."""
    return {
        'mean': np.mean(series),
        'median': np.median(series),
        'mode': stats.mode(series, keepdims=True)[0][0],
        'variance': np.var(series),
        'std_dev': np.std(series),
        'q1': np.quantile(series, 0.25),
        'q3': np.quantile(series, 0.75)
    }

def test_hypothesis(sample1, sample2, alpha=0.05):
    """Teste t de médias entre duas amostras."""
    t_stat, p_val = stats.ttest_ind(sample1, sample2)
    return {'t_stat': t_stat, 'p_value': p_val, 'reject_H0': p_val < alpha}

# =============================================================================
# DISTRIBUIÇÕES PROBABILÍSTICAS
# =============================================================================

def sample_distribution(dist_name, size=1000, **params):
    dists = {
        'normal': lambda: np.random.normal(params.get('mu', 0), params.get('sigma', 1), size),
        'bernoulli': lambda: np.random.binomial(1, params.get('p', 0.5), size),
        'binomial': lambda: np.random.binomial(params.get('n', 10), params.get('p', 0.5), size),
        'poisson': lambda: np.random.poisson(params.get('lam', 3), size),
        'uniform': lambda: np.random.uniform(params.get('a', 0), params.get('b', 1), size),
        'geometric': lambda: np.random.geometric(params.get('p', 0.5), size)
    }
    return dists[dist_name]()

# =============================================================================
# MÉTRICAS DE AVALIAÇÃO
# =============================================================================

def classification_metrics(y_true, y_pred, y_prob=None):
    """Calcula métricas comuns de classificação."""
    out = {
        'precision': metrics.precision_score(y_true, y_pred),
        'recall': metrics.recall_score(y_true, y_pred),
        'f1': metrics.f1_score(y_true, y_pred)
    }
    if y_prob is not None:
        fpr, tpr, _ = metrics.roc_curve(y_true, y_prob)
        auc = metrics.auc(fpr, tpr)
        out['auc'] = auc
        out['gini'] = 2 * auc - 1
        out['ks'] = max(tpr - fpr)
    return out

def regression_metrics(y_true, y_pred):
    """Métricas comuns de regressão."""
    return {
        'MAE': metrics.mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(metrics.mean_squared_error(y_true, y_pred)),
        'R2': metrics.r2_score(y_true, y_pred)
    }

# =============================================================================
# VALIDAÇÃO DE MODELOS
# =============================================================================

def kfold_validate(model, X, y, k=5):
    scores = model_selection.cross_val_score(model, X, y, cv=k, scoring='f1')
    return {'mean_f1': np.mean(scores), 'std_f1': np.std(scores)}

# =============================================================================
# PRÉ-PROCESSAMENTO
# =============================================================================

def handle_missing(df, strategy='mean'):
    """Preenche valores ausentes."""
    imp = preprocessing.SimpleImputer(strategy=strategy)
    return pd.DataFrame(imp.fit_transform(df), columns=df.columns)

def remove_outliers_iqr(df, factor=1.5):
    """Remove outliers via método do IQR."""
    clean = df.copy()
    for col in df.select_dtypes(include=np.number):
        q1, q3 = np.percentile(df[col], [25, 75])
        iqr = q3 - q1
        mask = (df[col] >= q1 - factor * iqr) & (df[col] <= q3 + factor * iqr)
        clean = clean[mask]
    return clean

def discretize_variable(series, bins=4):
    """Transforma variável contínua em discreta (quantis)."""
    return pd.qcut(series, q=bins, labels=False, duplicates='drop')

def run_pca(X, n_components=2):
    """Executa PCA."""
    pca = decomposition.PCA(n_components=n_components)
    Xp = pca.fit_transform(X)
    return Xp, pca.explained_variance_ratio_

# =============================================================================
# SELEÇÃO DE VARIÁVEIS
# =============================================================================

def feature_selection_lasso(X, y):
    """Seleciona variáveis via LassoCV."""
    model = LassoCV(cv=5).fit(X, y)
    selected = np.where(model.coef_ != 0)[0]
    return selected, model

# =============================================================================
# MODELOS DE CLASSIFICAÇÃO
# =============================================================================

def get_model(name):
    models = {
        'logistic': LogisticRegression(max_iter=1000),
        'rf': RandomForestClassifier(n_estimators=100),
        'gb': GradientBoostingClassifier(),
        'svm': SVC(probability=True),
        'knn': KNeighborsClassifier(),
        'gnb': GaussianNB(),
        'bnb': BernoulliNB(),
        'mlp': MLPClassifier(hidden_layer_sizes=(50,), max_iter=500)
    }
    return models[name]

# =============================================================================
# CLUSTERIZAÇÃO
# =============================================================================

def elbow_method(X, k_range=range(1, 10)):
    distortions = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42).fit(X)
        distortions.append(km.inertia_)
    plt.plot(k_range, distortions, marker='o')
    plt.title('Método do Cotovelo')
    plt.xlabel('Número de clusters')
    plt.ylabel('Distortion')
    plt.show()

def silhouette_analysis(X, k_range=range(2, 10)):
    scores = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42).fit(X)
        labels = km.labels_
        scores.append(silhouette_score(X, labels))
    plt.plot(k_range, scores, marker='o')
    plt.title('Análise de Silhueta')
    plt.xlabel('Número de clusters')
    plt.ylabel('Silhouette Score')
    plt.show()

# =============================================================================
# PESQUISA OPERACIONAL (LP e MIP)
# =============================================================================

def solve_lp(c, A, b, bounds=None):
    """Resolve um problema de programação linear via método Simplex."""
    res = linprog(c, A_ub=A, b_ub=b, bounds=bounds, method='highs')
    return res

try:
    import pulp
    def solve_mip(c, A, b, integer_idx=None):
        """Resolve MIP simples via PuLP."""
        n = len(c)
        x = [pulp.LpVariable(f"x{i}", lowBound=0, cat='Integer' if i in (integer_idx or []) else 'Continuous')
             for i in range(n)]
        prob = pulp.LpProblem('MIP', pulp.LpMinimize)
        prob += pulp.lpSum([c[i] * x[i] for i in range(n)])
        for i in range(len(A)):
            prob += pulp.lpSum([A[i][j] * x[j] for j in range(n)]) <= b[i]
        prob.solve()
        sol = [x[i].value() for i in range(n)]
        return {'solution': sol, 'objective': pulp.value(prob.objective), 'status': pulp.LpStatus[prob.status]}
except ImportError:
    def solve_mip(*args, **kwargs):
        raise ImportError("Instale pulp para usar o solver MIP.")

# =============================================================================
# EXEMPLO RÁPIDO
# =============================================================================
if __name__ == "__main__":
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer()
    X, y = data.data, data.target
    model = get_model('rf')
    X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.3, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(classification_metrics(y_test, y_pred, y_prob))
